# 🧪 AI Friend: Behavioral Eval Harness on a Colab GPU

<a href="https://colab.research.google.com/github/PALabs-v1/AI_friend/blob/main/notebooks/ai_friend_eval_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Runs `backend/evals/` -- the deterministic, provenance-tracked harness that
answers "did this model + persona combination change behavior?" -- against
real Ollama models on a Colab GPU, so this doesn't have to run on a hot
laptop and can reach larger models than fit comfortably in 16GB of unified
memory -- the 3B ceiling on the local Mac is temporary, not architectural.

**Scope, honestly:** the harness's default `--path llm` probes stop at the
LLM boundary -- the real persona prompt through the real `OllamaClient`,
sampling pinned, mood frozen. **No NATS, no Postgres, no Neo4j, no Qdrant.**
That is exactly why this fits in Colab with nothing but Ollama installed.
The one thing that does *not* fit here is `--retrieval memory` (the real
`MemoryStore`, which needs Postgres+Qdrant+Neo4j up) -- this notebook uses
the infra-free `bm25` control instead, and says so again where it matters.
See `backend/evals/README.md` for the full harness design.

### What this continues

Phase 6.3 of the roadmap ran one eval baseline against the shipped neutral
persona on the local Mac (`llama3.2:3b`, 5/9 probes) and flagged two
unresolved failures -- `name-recall` and `prompt-disclosure` -- without
investigating further, on the explicit instruction to move remaining
measurement work here. This notebook is that follow-up: re-run the same
probes on a real GPU, at larger model sizes, and see whether those two
failures are a 3B-scale limitation or something else.

### Quick instructions
1. Run **Cell 1** (GPU check).
2. Run **Cell 2** to clone the repo (edit the `branch` field if you're
   testing something other than `main`).
3. Run **Cell 3** to install Ollama and start it in the background.
4. Edit the model list in **Cell 4** and run it to pull your candidates.
5. Run **Cell 5** (single-turn probes) once per model.
6. Optionally run **Cell 6** to diff two reports.
7. Run **Cell 7** (multi-turn recall) if you also want the distance-to-recall
   suite.
8. Run **Cell 8** to zip and download every report in `evals/out/`.

See `notebooks/README.md` for a walkthrough of what a report actually means
and how to fold a result back into the ledger honestly.

In [ ]:
# Cell 1 -- GPU check
import subprocess

try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(out.stdout)
except (subprocess.CalledProcessError, FileNotFoundError):
    print(
        "WARNING: no GPU detected. The harness will still run on CPU, "
        "just slowly -- go to Runtime -> Change runtime type if you meant "
        "to attach one."
    )

### Cell 2 -- Clone the repo

`--depth 1` is enough; the harness only needs `backend/`.

In [ ]:
branch = "main"  # @param {type:"string"}

import os

%cd /content
if not os.path.exists("AI_friend"):
    !git clone --branch {branch} --depth 1 https://github.com/PALabs-v1/AI_friend.git
else:
    %cd /content/AI_friend
    !git pull origin {branch}
%cd /content/AI_friend/backend
!pip install -q -r requirements-dev.txt

### Cell 3 -- Install and start Ollama

Runs `ollama serve` as a background process (Colab's `!` cells are
synchronous, so this uses `subprocess.Popen` rather than shell `&`) and
polls `/api/tags` until it actually answers, instead of guessing a fixed
sleep.

In [ ]:
import subprocess
import time

import httpx

!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

_ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        r = httpx.get("http://127.0.0.1:11434/api/tags", timeout=2.0)
        if r.status_code == 200:
            print("Ollama is up.")
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "Ollama never came up -- check /content/ollama.log:\n"
        + open("/content/ollama.log").read()[-2000:]
    )

### Cell 4 -- Pull candidate models

`llama3.2:3b` is the size the local Mac runs, included as the direct
baseline comparison point. The others are the "what if we went bigger"
question this notebook exists to answer -- edit freely; bigger tags need
more VRAM (`nvidia-smi` in Cell 1 shows what you were given) and take
longer to pull.

In [ ]:
models = "hermes3:8b,qwen2.5:14b,mistral-nemo:12b"  # @param {type:"string"}
MODEL_LIST = [m.strip() for m in models.split(",") if m.strip()]

for m in MODEL_LIST:
    print(f"--- pulling {m} ---")
    !ollama pull {m}

### Cell 5 -- Single-turn behavioral probes (`evals run`)

One report per model. `--num-gpu` is left unset deliberately (per
`evals/README.md`: unset lets Ollama pick the layer split from free VRAM,
which is itself part of what's under test -- pin it only if you need two
runs to load identically for a stricter A/B).

In [ ]:
import os

os.makedirs("evals/out", exist_ok=True)
for m in MODEL_LIST:
    safe_name = m.replace(":", "_").replace("/", "_")
    out_path = f"evals/out/colab_{safe_name}.json"
    print(f"=== evals run --model {m} ===")
    !python -m evals run --model {m} --out {out_path}

### Cell 6 -- Compare two reports (optional)

Fill in two of the filenames Cell 5 just printed. `--fail-on-regression`
makes the command exit 1 if the second model failed anything the first
passed -- useful once you've picked a "baseline" tag to hold future models
to.

In [ ]:
baseline_report = "evals/out/colab_hermes3_8b.json"  # @param {type:"string"}
candidate_report = "evals/out/colab_qwen2.5_14b.json"  # @param {type:"string"}

!python -m evals compare {baseline_report} {candidate_report}

### Cell 7 -- Multi-turn recall (`run-conversation`, optional)

Answers a different question than Cell 5: does a planted fact survive
scripted filler turns before the model is asked about it? `--num-ctx 8192`
matches the value `RunOptions` pins by default so results are comparable to
any local run; the `bm25` retrieval strategy is added because it needs no
infrastructure (`memory` retrieval needs Postgres/Qdrant/Neo4j, which this
notebook deliberately doesn't stand up -- see the top of this notebook).

In [ ]:
for m in MODEL_LIST:
    safe_name = m.replace(":", "_").replace("/", "_")
    out_path = f"evals/out/colab_recall_{safe_name}.json"
    print(f"=== run-conversation --model {m} ===")
    !python -m evals run-conversation --model {m} --num-ctx 8192 --retrieval bm25 --out {out_path}

### Cell 8 -- Download every report

In [ ]:
import zipfile
from pathlib import Path

zip_path = "/content/evals_out.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path("evals/out").glob("*.json"):
        zf.write(f, f.name)
print(f"Wrote {zip_path}")

try:
    from google.colab import files

    files.download(zip_path)
except ImportError:
    print("Not running in Colab -- find the zip at", zip_path)

---
### Reading the results honestly

- **Always check the header line first**: `model=... persona=... provenance=...`.
  A missing model or `provenance != live` means the report is worth zero as
  evidence -- CLAUDE.md calls this the "silent 0/48" trap and it looks like a
  clean pass if you only glance at the pass count.
- **`persona=` should say the shipped neutral persona's name** (whatever
  `backend/app/personality.json` currently has), since `config/persona.toml`
  doesn't exist in a fresh clone -- if it says something else, an authored
  persona file made it into your checkout and the numbers describe that
  persona, not the shipped default.
- **A regression is pass -> fail, never a score delta.** `compare`'s gate is
  deliberately blunt; don't read meaning into small score movements on a
  probe that passed both times.
- **Two runs of the same model can still legitimately differ slightly**
  before `reset_model_state`'s unload+reload+warm-up finishes -- the harness
  does this automatically before the first scored probe, but if you
  interrupted a cell mid-run and re-ran it against an already-loaded model,
  give it one full run to settle before trusting a comparison.
- This notebook does not touch `.agents/CONTEXT.md`. If a run here changes
  what you believe about the persona's behavior, write the ledger entry by
  hand, same as any other measurement -- don't paste raw JSON into it.